# BBN on MovieLens-1M

## 0. Config and imports

In [11]:
import gc
import itertools
import json
import re
from collections import Counter
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


def set_seed(seed: int = 0):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [ ]:
CFG = {
    "work_dir": "../data",

    "embed_dim": 64,
    "tower_hidden": [256, 128, 64],
    "max_title_words": 2000,

    "batch_size": 1024,
    "num_workers": 0,
    "grad_clip": 1.0,

    "tune_fast": True,
    "tune_epochs": 3,
    "tune_val_fraction": 1.0,
    "skip_tune_if_cached": True,
    "cache_path": "movielens_bbn_best_hparams.json",

    "final_epochs": 40,
    "seeds": [0, 1, 2, 3, 4],

    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 1024,
    "head_fraction": 0.20,

    # BBN-specific
    "resample_head_fraction": 0.20,
    "bbn_alpha_max": 0.5,

    "seed": 0,
}

set_seed(CFG["seed"])
print(f"Final: {len(CFG['seeds'])} seeds × {CFG['final_epochs']} epochs")

Final: 5 seeds × 40 epochs


## 1. Load MovieLens-1M

In [13]:
GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]


def read_ml1m(raw: Path):
    users = pd.read_csv(
        raw / "users.dat",
        sep="::",
        engine="python",
        names=["user_id", "gender", "age", "occupation", "zip"],
        encoding="latin-1",
    )

    ratings = pd.read_csv(
        raw / "ratings.dat",
        sep="::",
        engine="python",
        names=["user_id", "item_id", "rating", "timestamp"],
        encoding="latin-1",
    )

    ratings = ratings[ratings["rating"] > 0].copy()

    movies = []
    with open(raw / "movies.dat", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("::")
            mid = int(parts[0])
            title = parts[1]
            genres = parts[2].split("|") if len(parts) > 2 else []

            year = 1995
            if title.endswith(")") and "(" in title:
                try:
                    year = int(title[title.rfind("(") + 1 : -1])
                except ValueError:
                    pass

            movies.append({
                "item_id": mid,
                "title": title,
                "genres": genres,
                "year": year,
            })

    return users, pd.DataFrame(movies), ratings


RAW_DIR = Path("/kaggle/input/datasets/ugaliok/movielens1m/data")
users_df, movies_df, ratings = read_ml1m(RAW_DIR)

item_ids = sorted(ratings["item_id"].unique())
uid_map = {u: i for i, u in enumerate(users_df["user_id"].tolist())}
iid_map = {m: i for i, m in enumerate(item_ids)}

NUM_USERS = len(uid_map)
NUM_ITEMS = len(iid_map)

print(f"users={NUM_USERS:,} items={NUM_ITEMS:,} ratings={len(ratings):,}")

users=6,040 items=3,706 ratings=1,000,209


## 2. Feature engineering and leave-one-out split

In [14]:
def tokenize_title(title: str):
    if "(" in title:
        title = title.rsplit("(", 1)[0]
    return re.findall(r"[a-z0-9]+", title.lower())


def build_title_bow(titles, max_words: int):
    counter = Counter()
    per_title = []

    for t in titles:
        toks = tokenize_title(t)
        per_title.append(toks)
        counter.update(toks)

    vocab = [w for w, _ in counter.most_common(max_words)]
    w2i = {w: i for i, w in enumerate(vocab)}

    mat = np.zeros((len(titles), len(vocab)), dtype=np.float32)

    for r, toks in enumerate(per_title):
        for tok in toks:
            j = w2i.get(tok)
            if j is not None:
                mat[r, j] = 1.0

    return mat


# ---------- item features ----------
genre_to_idx = {g: i for i, g in enumerate(GENRES)}

genre_matrix = []
years = []
titles = []

for mid in item_ids:
    row = movies_df.loc[movies_df["item_id"] == mid].iloc[0]

    gvec = np.zeros(len(GENRES), dtype=np.float32)
    for g in row["genres"]:
        if g in genre_to_idx:
            gvec[genre_to_idx[g]] = 1.0

    genre_matrix.append(gvec)
    years.append(row["year"])
    titles.append(row["title"])

years = np.asarray(years, dtype=np.float32)
years = (years - years.mean()) / (years.std() + 1e-6)

title_bow = build_title_bow(titles, CFG["max_title_words"])

ITEM_GEN = np.hstack([
    np.stack(genre_matrix),
    years.reshape(-1, 1),
    title_bow,
]).astype(np.float32)

ITEM_GEN = (ITEM_GEN - ITEM_GEN.mean(axis=0, keepdims=True)) / (ITEM_GEN.std(axis=0, keepdims=True) + 1e-6)
ITEM_GEN = ITEM_GEN.astype(np.float32)
ITEM_GEN_DIM = ITEM_GEN.shape[1]


# ---------- user features ----------
gender_map = {g: i for i, g in enumerate(sorted(users_df["gender"].unique()))}
age_map = {a: i for i, a in enumerate(sorted(users_df["age"].unique()))}
occupation_map = {o: i for i, o in enumerate(sorted(users_df["occupation"].unique()))}
zip_map = {z: i for i, z in enumerate(sorted(users_df["zip"].unique()))}

USER_GEN = np.stack([
    users_df["gender"].map(gender_map),
    users_df["age"].map(age_map),
    users_df["occupation"].map(occupation_map),
    users_df["zip"].map(zip_map),
], axis=1).astype(np.int64)

USER_GEN_SIZES = [
    len(gender_map),
    len(age_map),
    len(occupation_map),
    len(zip_map),
]

print("ITEM_GEN_DIM:", ITEM_GEN_DIM)
print("USER_GEN_SIZES:", USER_GEN_SIZES)


# ---------- leave-one-out split ----------
train_pairs = []
val_u, val_i = [], []
test_u, test_i = [], []

train_user_items = [set() for _ in range(NUM_USERS)]

for uid, grp in ratings.groupby("user_id"):
    if uid not in uid_map:
        continue

    u = uid_map[uid]
    item_seq = [
        iid_map[i]
        for i in grp.sort_values("timestamp")["item_id"].tolist()
        if i in iid_map
    ]

    if len(item_seq) < 3:
        continue

    for it in item_seq[:-2]:
        train_pairs.append((u, it))
        train_user_items[u].add(it)

    val_u.append(u)
    val_i.append(item_seq[-2])

    test_u.append(u)
    test_i.append(item_seq[-1])

train_pairs = np.asarray(train_pairs, dtype=np.int64)
val_u = np.asarray(val_u, dtype=np.int64)
val_i = np.asarray(val_i, dtype=np.int64)
test_u = np.asarray(test_u, dtype=np.int64)
test_i = np.asarray(test_i, dtype=np.int64)

print(f"train={len(train_pairs):,} val={len(val_u):,} test={len(test_u):,}")

known_val = [set(s) for s in train_user_items]

known_test = [set(s) for s in train_user_items]
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)

order = np.argsort(-item_freq)
n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))

head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,} tail={(~head_mask).sum():,}")
print(f"test_head_share={head_mask[test_i].mean():.4f}")
print(f"test_tail_share={(~head_mask[test_i]).mean():.4f}")

ITEM_GEN_DIM: 2019
USER_GEN_SIZES: [2, 7, 21, 3439]
train=988,129 val=6,040 test=6,040
head=741 tail=2,965
test_head_share=0.6022
test_tail_share=0.3978


## 3. Model

In [15]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.0):
        super().__init__()

        layers = []
        d = in_dim

        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            d = h

        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UserGenEnc(nn.Module):
    def __init__(self, sizes: list[int], dim: int = 16):
        super().__init__()

        self.embs = nn.ModuleList([
            nn.Embedding(n, dim)
            for n in sizes
        ])

        self.out_dim = len(sizes) * dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat(
            [emb(x[:, i].long()) for i, emb in enumerate(self.embs)],
            dim=-1,
        )


class TwoTower(nn.Module):
    def __init__(self, dropout: float = 0.0):
        super().__init__()

        h = CFG["tower_hidden"]
        ed = CFG["embed_dim"]

        self.user_id = nn.Embedding(NUM_USERS, ed)
        self.user_gen = UserGenEnc(USER_GEN_SIZES, dim=16)
        self.user_mlp = MLP(ed + self.user_gen.out_dim, h, dropout)
        self.user_ln = nn.LayerNorm(self.user_mlp.out_dim)

        self.item_id = nn.Embedding(NUM_ITEMS, ed)
        self.item_mlp = MLP(ed + ITEM_GEN_DIM, h, dropout)
        self.item_ln = nn.LayerNorm(self.item_mlp.out_dim)

        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def user_vec(self, u: torch.Tensor, ug: torch.Tensor) -> torch.Tensor:
        x = torch.cat([self.user_id(u), self.user_gen(ug)], dim=-1)
        x = self.user_mlp(x)
        x = self.user_ln(x)
        return x

    def item_vec(self, i: torch.Tensor, ig: torch.Tensor) -> torch.Tensor:
        x = torch.cat([self.item_id(i), ig], dim=-1)
        x = self.item_mlp(x)
        x = self.item_ln(x)
        return x


def inbatch_softmax_loss(user_vecs: torch.Tensor, item_vecs: torch.Tensor) -> torch.Tensor:
    u = F.normalize(user_vecs, dim=-1, eps=1e-6)
    v = F.normalize(item_vecs, dim=-1, eps=1e-6)

    logits = u @ v.T
    logits = torch.clamp(logits, -20.0, 20.0)

    if not torch.isfinite(logits).all():
        raise RuntimeError("NaN/Inf in logits")

    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


ug_t = torch.from_numpy(USER_GEN).long().to(device)
ig_t = torch.from_numpy(ITEM_GEN).float().to(device)

train_loader = DataLoader(
    PairDataset(train_pairs),
    batch_size=CFG["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

print("Model helpers ready.")

Model helpers ready.


## 4. Evaluation

In [16]:
@torch.no_grad()
def evaluate_full_corpus(
    model: nn.Module,
    users: np.ndarray,
    true_items: np.ndarray,
    known_user_items: List[set],
    head_mask: np.ndarray,
    ks: List[int],
    item_freq: np.ndarray,
    user_batch_size: int = 128,
    item_batch_size: int = 1024,
):
    model.eval()

    ranks_all = []
    ranks_head = []
    ranks_tail = []

    max_k = max(ks)
    recommended_by_k = {k: [] for k in ks}

    item_vectors = []

    for s in tqdm(range(0, NUM_ITEMS, item_batch_size), desc="item vectors", leave=False):
        e = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)

        v = model.item_vec(idx, ig_t[idx])
        v = F.normalize(v, dim=-1, eps=1e-6)

        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors on item batch {s}:{e}")

        item_vectors.append(v.cpu())

    item_vectors = torch.cat(item_vectors, dim=0).to(device)

    for start in tqdm(range(0, len(users), user_batch_size), desc="eval users", leave=False):
        end = min(start + user_batch_size, len(users))

        bu = users[start:end]
        bi = true_items[start:end]

        ut = torch.tensor(bu, device=device, dtype=torch.long)

        uvec = model.user_vec(ut, ug_t[ut])
        uvec = F.normalize(uvec, dim=-1, eps=1e-6)

        if not torch.isfinite(uvec).all():
            raise RuntimeError(f"Non-finite user vectors on user batch {start}:{end}")

        scores = (uvec @ item_vectors.T).cpu().numpy()

        if not np.isfinite(scores).all():
            bad = np.sum(~np.isfinite(scores))
            raise RuntimeError(f"Non-finite scores in user batch {start}:{end}: {bad}")

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            u = int(u)
            true_i = int(true_i)

            srow = scores[row].copy()

            for it in known_user_items[u]:
                if it != true_i:
                    srow[it] = -1e9

            true_score = srow[true_i]
            eps = 1e-12

            num_greater = int((srow > true_score + eps).sum())
            num_tied = int((np.abs(srow - true_score) <= eps).sum())
            rank = num_greater + num_tied - 1

            ranks_all.append(rank)

            if head_mask[true_i]:
                ranks_head.append(rank)
            else:
                ranks_tail.append(rank)

            if max_k < len(srow):
                top_unsorted = np.argpartition(-srow, max_k - 1)[:max_k]
                top_idx = top_unsorted[np.argsort(-srow[top_unsorted])]
            else:
                top_idx = np.argsort(-srow)

            for k in ks:
                recommended_by_k[k].append(top_idx[:k].astype(np.int64))

    def agg_accuracy(ranks: List[int]) -> Dict[str, float]:
        a = np.asarray(ranks, dtype=np.int64)
        out = {}

        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = np.nan
                out[f"NDCG@{k}"] = np.nan
            else:
                hits = a < k
                out[f"HR@{k}"] = 100.0 * hits.mean()
                out[f"NDCG@{k}"] = 100.0 * np.where(
                    hits,
                    1.0 / np.log2(a + 2),
                    0.0,
                ).mean()

        return out

    tail_mask = ~head_mask
    num_tail_items = int(tail_mask.sum())

    popularity = np.log1p(item_freq.astype(np.float64))

    long_tail = {}

    for k in ks:
        recs = np.vstack(recommended_by_k[k])
        unique_recs = np.unique(recs)

        catalog_coverage = len(unique_recs) / NUM_ITEMS

        if num_tail_items > 0:
            tail_coverage = np.sum(tail_mask[unique_recs]) / num_tail_items
        else:
            tail_coverage = np.nan

        avg_popularity = popularity[recs].mean()
        tail_ratio = tail_mask[recs].mean()

        n_users_eval = recs.shape[0]

        if n_users_eval <= 1:
            personalization = np.nan
        else:
            exposure_counts = np.bincount(recs.reshape(-1), minlength=NUM_ITEMS)
            pairwise_intersections = np.sum(exposure_counts * (exposure_counts - 1) / 2)

            num_user_pairs = n_users_eval * (n_users_eval - 1) / 2
            avg_overlap = pairwise_intersections / num_user_pairs

            personalization = 1.0 - avg_overlap / k

        long_tail[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail[f"TailCoverage@{k}"] = 100.0 * tail_coverage
        long_tail[f"AvgPopularity@{k}"] = float(avg_popularity)
        long_tail[f"TailRatio@{k}"] = 100.0 * tail_ratio
        long_tail[f"Personalization@{k}"] = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head": agg_accuracy(ranks_head),
        "tail": agg_accuracy(ranks_tail),
        "long_tail": long_tail,
        "counts": {
            "overall": len(ranks_all),
            "head": len(ranks_head),
            "tail": len(ranks_tail),
        },
    }


def print_metrics(metrics: Dict):
    print("counts:", metrics.get("counts", {}))

    for split in ("overall", "head", "tail"):
        print(f"[{split}]", metrics[split])

    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])

## 5. BBN data and $\alpha$-schedule

In [ ]:
def build_rebalanced_pairs(
    train_pairs: np.ndarray,
    item_freq: np.ndarray,
    head_fraction: float,
    rng_seed: int = 0,
) -> np.ndarray:
    n_items = len(item_freq)
    order = np.argsort(-item_freq)
    n_head = max(1, int(n_items * head_fraction))

    local_head_mask = np.zeros(n_items, dtype=bool)
    local_head_mask[order[:n_head]] = True

    tail_freq = item_freq[~local_head_mask]
    target_freq = int(tail_freq.max()) if tail_freq.max() > 0 else 1

    rng = np.random.default_rng(rng_seed)

    item_to_rows: dict[int, list[int]] = {}
    for row_idx, (u, i) in enumerate(train_pairs):
        item_to_rows.setdefault(int(i), []).append(row_idx)

    kept = []
    for item_idx, row_indices in item_to_rows.items():
        if local_head_mask[item_idx]:
            n_keep = min(target_freq, len(row_indices))
            chosen = rng.choice(row_indices, size=n_keep, replace=False)
            kept.extend(chosen.tolist())
        else:
            kept.extend(row_indices)

    return train_pairs[sorted(kept)]


rebalanced_pairs = build_rebalanced_pairs(
    train_pairs=train_pairs,
    item_freq=item_freq,
    head_fraction=CFG["resample_head_fraction"],
    rng_seed=CFG["seed"],
)

rb_freq = np.bincount(rebalanced_pairs[:, 1], minlength=NUM_ITEMS)

print(f"Original  train pairs: {len(train_pairs):,}")
print(f"Rebalanced train pairs: {len(rebalanced_pairs):,}")
print(
    f"Original  head/tail freq ratio: "
    f"{item_freq[head_mask].mean():.1f} / {item_freq[~head_mask].mean():.1f}"
)
print(
    f"Rebalanced head/tail freq ratio: "
    f"{rb_freq[head_mask].mean():.1f} / {rb_freq[~head_mask].mean():.1f}"
)

rebalanced_loader = DataLoader(
    PairDataset(rebalanced_pairs),
    batch_size=CFG["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)


def alpha_schedule(epoch: int, total_epochs: int, alpha_max: float) -> float:
    if total_epochs <= 1:
        return alpha_max
    return alpha_max * (epoch - 1) / (total_epochs - 1)


def bbn_loss(
    model: nn.Module,
    users_cl: torch.Tensor,
    items_cl: torch.Tensor,
    users_rb: torch.Tensor,
    items_rb: torch.Tensor,
    alpha: float,
) -> torch.Tensor:
    # L = (1 - alpha) * L_CL + alpha * L_RB
    loss_cl = inbatch_softmax_loss(
        model.user_vec(users_cl, ug_t[users_cl]),
        model.item_vec(items_cl, ig_t[items_cl]),
    )

    loss_rb = inbatch_softmax_loss(
        model.user_vec(users_rb, ug_t[users_rb]),
        model.item_vec(items_rb, ig_t[items_rb]),
    )

    return (1.0 - alpha) * loss_cl + alpha * loss_rb

Original  train pairs: 988,129
Rebalanced train pairs: 657,176
Original  head/tail freq ratio: 869.6 / 115.9
Rebalanced head/tail freq ratio: 423.0 / 115.9
rebalanced_loader batches/epoch: 641
BBN alpha schedule: 0.0 → 0.5 over 40 epochs


## 6. Training helper

In [18]:
def train_one_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    total_epochs: int,
    desc: str,
    alpha_max: float,
) -> float:
    model.train()

    alpha = alpha_schedule(epoch, total_epochs, alpha_max)

    total_loss = 0.0
    total_n = 0

    rb_iter = itertools.cycle(rebalanced_loader)

    pbar = tqdm(
        train_loader,
        desc=f"{desc} {epoch}/{total_epochs} α={alpha:.3f}",
        leave=False,
    )

    for users_cl, items_cl in pbar:
        users_rb, items_rb = next(rb_iter)

        users_cl = users_cl.to(device, non_blocking=True)
        items_cl = items_cl.to(device, non_blocking=True)
        users_rb = users_rb.to(device, non_blocking=True)
        items_rb = items_rb.to(device, non_blocking=True)

        loss = bbn_loss(
            model=model,
            users_cl=users_cl,
            items_cl=items_cl,
            users_rb=users_rb,
            items_rb=items_rb,
            alpha=alpha,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss.item()}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])

        optimizer.step()

        bs = users_cl.size(0)
        total_loss += loss.item() * bs
        total_n += bs

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(total_n, 1)

## 7. Grid search

In [19]:
LR_GRID = [0.1, 0.01, 0.001, 0.0001]
DROPOUT_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]
WD_GRID = [0.0, 0.1, 0.01, 0.001]
ALPHA_MAX_GRID = [0.25, 0.5]

combos = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID, ALPHA_MAX_GRID))
k_main = CFG["eval_k"][-1]

cache_path = Path(CFG["cache_path"])
leaderboard_csv_path = cache_path.with_suffix(".leaderboard.csv")

if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text())
    print(f"Loaded cached hparams: {best_hp}")

    leaderboard_df = pd.read_csv(leaderboard_csv_path) if leaderboard_csv_path.exists() else None

else:
    frac = float(CFG.get("tune_val_fraction", 1.0))

    if frac < 1.0 and CFG["tune_fast"]:
        n_tune = max(1, int(len(val_u) * frac))
        idx = np.random.default_rng(42).choice(len(val_u), n_tune, replace=False)
        val_u_t, val_i_t = val_u[idx], val_i[idx]
    else:
        val_u_t, val_i_t = val_u, val_i

    tune_ep = CFG["tune_epochs"] if CFG["tune_fast"] else CFG["final_epochs"]

    print(
        f"Tuning BBN {len(combos)} trials × {tune_ep} ep "
        f"(val subset: {len(val_u_t):,}/{len(val_u):,})"
    )

    best_hr = -1.0
    best_hp = None
    leaderboard = []

    for ti, (lr, dr, wd, alpha_max) in enumerate(combos, 1):
        set_seed(CFG["seed"])

        m = TwoTower(dropout=dr).to(device)

        opt = torch.optim.Adam(
            m.parameters(),
            lr=lr,
            weight_decay=wd,
        )

        status = "ok"
        error_message = ""
        avg_loss = np.nan
        met = None
        hr = -1.0

        try:
            for ep in range(1, tune_ep + 1):
                avg_loss = train_one_epoch(
                    model=m,
                    optimizer=opt,
                    epoch=ep,
                    total_epochs=tune_ep,
                    desc=f"BBN trial{ti}",
                    alpha_max=alpha_max,
                )
                print(f"  BBN trial{ti} ep{ep} loss={avg_loss:.4f}")

            met = evaluate_full_corpus(
                model=m,
                users=val_u_t,
                true_items=val_i_t,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )

            hr = met["overall"][f"HR@{k_main}"]

            if not np.isfinite(hr):
                raise RuntimeError(f"Non-finite HR: {hr}")

        except RuntimeError as e:
            status = "failed"
            error_message = str(e)
            print(f"  BBN trial {ti} FAILED: {e}")

        row = {
            "trial": ti,
            "status": status,
            "error": error_message,
            "lr": lr,
            "dropout": dr,
            "weight_decay": wd,
            "alpha_max": alpha_max,
            "tune_epochs": tune_ep,
            "val_subset_size": len(val_u_t),
            "val_full_size": len(val_u),
            "final_train_loss": avg_loss,
            f"val_HR@{k_main}": hr,
        }

        if met is not None:
            for split in ("overall", "head", "tail"):
                for key, value in met[split].items():
                    row[f"val_{split}_{key}"] = value

            if "long_tail" in met:
                for key, value in met["long_tail"].items():
                    row[f"val_{key}"] = value

        leaderboard.append(row)

        print(
            f"  BBN trial {ti:3d}/{len(combos)} "
            f"lr={lr:.0e} dr={dr} wd={wd:.0e} alpha_max={alpha_max} "
            f"-> val HR@{k_main}={hr:.2f}%"
        )

        if status == "ok" and hr > best_hr:
            best_hr = hr
            best_hp = {
                "lr": lr,
                "dropout": dr,
                "weight_decay": wd,
                "alpha_max": float(alpha_max),
                f"val_HR@{k_main}": hr,
            }

        del m
        del opt
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    leaderboard_df = pd.DataFrame(leaderboard).sort_values(
        f"val_HR@{k_main}",
        ascending=False,
        na_position="last",
    )

    leaderboard_df.to_csv(leaderboard_csv_path, index=False)

    if best_hp is None:
        raise RuntimeError("No successful grid-search trial.")

    cache_path.write_text(json.dumps(best_hp, indent=2))

    print(f"\nBest val HR@{k_main}={best_hr:.2f}% -> {best_hp}")
    print(f"Saved best hparams: {cache_path}")
    print(f"Saved leaderboard: {leaderboard_csv_path}")

leaderboard_df.head(10) if leaderboard_df is not None else None

Tuning BBN 160 trials × 3 ep (val subset: 6,040/6,040)


BBN trial1 1/3 α=0.000:   0%|          | 0/964 [00:00<?, ?it/s]

  BBN trial1 ep1 loss=6.8784


BBN trial1 2/3 α=0.125:   0%|          | 0/964 [00:00<?, ?it/s]

  BBN trial1 ep2 loss=6.8416


...

  BBN trial160 ep3 loss=6.9362


item vectors:   0%|          | 0/4 [00:00<?, ?it/s]

eval users:   0%|          | 0/48 [00:00<?, ?it/s]

  BBN trial 160/160 lr=1e-04 dr=0.9 wd=1e-03 alpha_max=0.5 -> val HR@50=0.98%

Best val HR@50=10.70% -> {'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0, 'alpha_max': 0.25, 'val_HR@50': np.float64(10.695364238410596)}
Saved best hparams: movielens_bbn_best_hparams.json
Saved leaderboard: movielens_bbn_best_hparams.leaderboard.csv


,trial,status,error,lr,dropout,weight_decay,alpha_max,tune_epochs,val_subset_size,val_full_size,...,val_CatalogCoverage@10,val_TailCoverage@10,val_AvgPopularity@10,val_TailRatio@10,val_Personalization@10,val_CatalogCoverage@50,val_TailCoverage@50,val_AvgPopularity@50,val_TailRatio@50,val_Personalization@50
80,81,ok,,0.0010,0.1,0.000,0.25,3,6040,6040,...,96.654074,95.919056,5.109211,61.475166,99.278428,100.000000,100.000000,5.000601,66.195033,97.284638
81,82,ok,,0.0010,0.1,0.000,0.50,3,6040,6040,...,97.895305,97.436762,5.046903,63.385762,99.345168,100.000000,100.000000,4.945904,67.866887,97.458678
41,42,ok,,0.0100,0.1,0.000,0.50,3,6040,6040,...,91.851052,90.084317,5.362692,54.192053,99.251874,99.973017,99.966273,5.263375,58.690066,97.181124
40,41,ok,,0.0100,0.1,0.000,0.25,3,6040,6040,...,91.041554,88.971332,5.309817,54.072848,99.171557,99.973017,99.966273,5.262977,57.704305,96.969411
87,88,ok,,0.0010,0.1,0.001,0.50,3,6040,6040,...,85.752833,83.709949,5.276669,56.910596,98.914040,99.703184,99.662732,5.129364,62.315894,96.577362
86,87,ok,,0.0010,0.1,0.001,0.25,3,6040,6040,...,82.919590,80.370995,5.205644,56.832781,98.838698,99.271452,99.123103,5.065991,61.896358,96.346594
1,2,ok,,0.1000,0.1,0.000,0.50,3,6040,6040,...,93.497032,92.613828,4.792791,67.423841,99.270943,100.000000,100.000000,4.746016,70.687748,97.463892
126,127,ok,,0.0001,0.1,0.001,0.25,3,6040,6040,...,81.111711,79.156830,4.857391,68.298013,98.989157,99.541284,99.527825,4.715751,73.650993,97.294752
140,141,ok,,0.0001,0.5,0.010,0.25,3,6040,6040,...,1.699946,0.370995,7.073452,14.152318,43.705711,4.991905,2.563238,6.528339,21.565894,27.148916
0,1,ok,,0.1000,0.1,0.000,0.25,3,6040,6040,...,91.122504,89.848229,4.743736,67.538079,99.298393,100.000000,100.000000,4.715989,70.034106,97.445850


## 8. Final training over 5 seeds

In [ ]:
all_rows = []
all_test = []

print("Using best_hp:", best_hp)

for seed in CFG["seeds"]:
    print("\n" + "=" * 80)
    print(f"MovieLens BBN seed {seed}")
    print("=" * 80)

    set_seed(seed)

    model = TwoTower(dropout=best_hp["dropout"]).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_hp["lr"],
        weight_decay=best_hp["weight_decay"],
    )

    alpha_max = float(best_hp["alpha_max"])

    best_val_hr50 = -1.0
    best_state = None

    for epoch in range(1, CFG["final_epochs"] + 1):
        avg_loss = train_one_epoch(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            total_epochs=CFG["final_epochs"],
            desc=f"BBN seed{seed}",
            alpha_max=alpha_max,
        )

        alpha = alpha_schedule(epoch, CFG["final_epochs"], alpha_max)
        print(f"seed {seed} epoch {epoch}: train loss = {avg_loss:.4f}  α={alpha:.3f}")

        val_metrics = evaluate_full_corpus(
            model=model,
            users=val_u,
            true_items=val_i,
            known_user_items=known_val,
            head_mask=head_mask,
            ks=CFG["eval_k"],
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )

        hr50 = val_metrics["overall"].get("HR@50", -1.0)

        if hr50 > best_val_hr50:
            best_val_hr50 = hr50
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            print(f"new best val HR@50 = {best_val_hr50:.4f}")

    print(f"seed {seed} best val HR@50: {best_val_hr50:.4f}")

    assert best_state is not None

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    test_metrics = evaluate_full_corpus(
        model=model,
        users=test_u,
        true_items=test_i,
        known_user_items=known_test,
        head_mask=head_mask,
        ks=CFG["eval_k"],
        item_freq=item_freq,
        user_batch_size=CFG["eval_batch_users"],
        item_batch_size=CFG["eval_item_batch"],
    )

    print("TEST METRICS")
    print_metrics(test_metrics)

    all_test.append(test_metrics)

    row = {
        "method": "BBN",
        "seed": seed,
        "lr": best_hp["lr"],
        "dropout": best_hp["dropout"],
        "weight_decay": best_hp["weight_decay"],
        "alpha_max": best_hp["alpha_max"],
        "best_val_HR@50": best_val_hr50,
    }

    for split in ("overall", "head", "tail"):
        for key, value in test_metrics[split].items():
            row[f"test_{split}_{key}"] = value

    if "long_tail" in test_metrics:
        for key, value in test_metrics["long_tail"].items():
            row[f"test_{key}"] = value

    if "counts" in test_metrics:
        for key, value in test_metrics["counts"].items():
            row[f"test_count_{key}"] = value

    all_rows.append(row)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_rows)
metrics_df

Using best_hp: {'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0, 'alpha_max': 0.25, 'val_HR@50': np.float64(10.695364238410596)}

MovieLens BBN seed 0


BBN seed0 1/40 α=0.000:   0%|          | 0/964 [00:00<?, ?it/s]

seed 0 epoch 1: train loss = 6.8158  α=0.000


item vectors:   0%|          | 0/4 [00:00<?, ?it/s]

eval users:   0%|          | 0/48 [00:00<?, ?it/s]

...

BBN seed4 12/40 α=0.071:   0%|          | 0/964 [00:00<?, ?it/s]

seed 4 epoch 12: train loss = 6.6964  α=0.071


item vectors:   0%|          | 0/4 [00:00<?, ?it/s]

eval users:   0%|          | 0/48 [00:00<?, ?it/s]

BBN seed4 13/40 α=0.077:   0%|          | 0/964 [00:00<?, ?it/s]

## 9. Compact final summary

In [ ]:
def make_movielens_summary_table(
    metrics_df: pd.DataFrame,
    method_name: str,
    save_path: str,
) -> pd.DataFrame:
    selected_metrics = [
        "test_overall_HR@10",
        "test_overall_NDCG@10",
        "test_overall_HR@50",
        "test_overall_NDCG@50",
        "test_head_HR@50",
        "test_head_NDCG@50",
        "test_tail_HR@50",
        "test_tail_NDCG@50",
        "test_CatalogCoverage@50",
        "test_TailCoverage@50",
        "test_AvgPopularity@50",
        "test_TailRatio@50",
        "test_Personalization@50",
    ]

    row = {
        "method": method_name,
        "num_seeds": metrics_df["seed"].nunique() if "seed" in metrics_df.columns else len(metrics_df),
        "head_fraction": CFG["head_fraction"],
        "head_share": f"{100 * CFG['head_fraction']:.1f}%",
        "num_head_items": int(head_mask.sum()),
        "num_tail_items": int((~head_mask).sum()),
        "test_head_share": f"{100 * float(head_mask[test_i].mean()):.2f}%",
        "test_tail_share": f"{100 * float((~head_mask[test_i]).mean()):.2f}%",
    }

    for hp_col in ["lr", "dropout", "weight_decay", "alpha_max"]:
        if hp_col in metrics_df.columns:
            vals = metrics_df[hp_col].dropna().unique()
            if len(vals) == 1:
                row[hp_col] = vals[0]

    if "best_val_HR@50" in metrics_df.columns:
        vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
        if len(vals) > 0:
            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            row["best_val_HR@50"] = f"{mean:.2f} ± {std:.2f}"

    for metric in selected_metrics:
        if metric not in metrics_df.columns:
            continue

        vals = metrics_df[metric].dropna().to_numpy(dtype=float)
        out_name = metric.replace("test_", "")

        if len(vals) == 0:
            row[out_name] = "nan"
            continue

        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

        if "AvgPopularity" in out_name:
            row[out_name] = f"{mean:.4f} ± {std:.4f}"
        else:
            row[out_name] = f"{mean:.2f} ± {std:.2f}"

    summary_df = pd.DataFrame([row])

    summary_df.to_csv(save_path, index=False)
    print(f"saved: {save_path}")

    return summary_df

In [ ]:
summary_df = make_movielens_summary_table(
    metrics_df=metrics_df,
    method_name="BBN",
    save_path="movielens_bbn_summary.csv",
)

summary_df